# ETL — HR Employee Attrition
**Origem:** `HR-Employee-Attrition.csv`  
**Destino:** arquivos `.csv` limpos prontos para carga no Power BI  
**Etapas:** Extração → Validação → Transformação → Enriquecimento → Exportação

---

## 0. Dependências

In [20]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Extração

Leitura do arquivo bruto e snapshot inicial dos dados.

In [21]:
#  Caminho do arquivo 
RAW_FILE   = "HR-Employee-Attrition.csv"   # ajuste o path se necessário
OUTPUT_DIR = "output_pbi"                  # pasta de saída para o Power BI

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_raw = pd.read_csv(RAW_FILE)

print(f"Shape: {df_raw.shape[0]:,} linhas  ×  {df_raw.shape[1]} colunas")
print()
df_raw.head(3)


Shape: 1,470 linhas  ×  35 colunas



,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0


In [22]:
#  2.5 Tipos e nulos 
info = pd.DataFrame({
    "dtype"       : df_raw.dtypes,
    "nulos"       : df_raw.isnull().sum(),
    "nulos_%"     : (df_raw.isnull().mean() * 100).round(2),
    "únicos"      : df_raw.nunique(),
    "amostra"     : df_raw.iloc[0],
})
print(info.to_string())


                          dtype  nulos  nulos_%  únicos          amostra
Age                       int64      0     0.00      43               41
Attrition                   str      0     0.00       2              Yes
BusinessTravel              str      0     0.00       3    Travel_Rarely
DailyRate                 int64      0     0.00     886             1102
Department                  str      0     0.00       3            Sales
DistanceFromHome          int64      0     0.00      29                1
Education                 int64      0     0.00       5                2
EducationField              str      0     0.00       6    Life Sciences
EmployeeCount             int64      0     0.00       1                1
EmployeeNumber            int64      0     0.00    1470                1
EnvironmentSatisfaction   int64      0     0.00       4                2
Gender                      str      0     0.00       2           Female
HourlyRate                int64      0     0.00    

## 2. Validação de qualidade

Checagens automáticas antes de qualquer transformação.  
Falhas críticas interrompem o pipeline; avisos são registrados.

In [23]:
import sys

erros   = []
avisos  = []

#  2.1 Colunas obrigatórias 
COLUNAS_OBRIGATORIAS = [
    "EmployeeNumber", "Age", "Attrition", "Department", "JobRole",
    "MonthlyIncome", "OverTime", "YearsAtCompany",
]
faltando = [c for c in COLUNAS_OBRIGATORIAS if c not in df_raw.columns]
if faltando:
    erros.append(f"Colunas obrigatórias ausentes: {faltando}")

#  2.2 Nulos 
nulos = df_raw.isnull().sum()
cols_com_nulos = nulos[nulos > 0]
if not cols_com_nulos.empty:
    avisos.append(f"Colunas com nulos:\n{cols_com_nulos.to_string()}")

#  2.3 EmployeeNumber duplicado 
dupl = df_raw["EmployeeNumber"].duplicated().sum()
if dupl > 0:
    erros.append(f"EmployeeNumber com {dupl} duplicata(s) — verifique a fonte.")

# 2.4 Faixas esperadas 
checks_range = {
    "Age"               : (18, 65),
    "MonthlyIncome"     : (1000, 25000),
    "PercentSalaryHike" : (0, 100),
    "TotalWorkingYears" : (0, 50),
    "YearsAtCompany"    : (0, 50),
}
for col, (lo, hi) in checks_range.items():
    if col in df_raw.columns:
        fora = df_raw[(df_raw[col] < lo) | (df_raw[col] > hi)]
        if not fora.empty:
            avisos.append(f"{col}: {len(fora)} valor(es) fora de [{lo}, {hi}]")

CATEGORIAS_ESPERADAS = {
    "Attrition"     : {"Yes", "No"},
    "OverTime"      : {"Yes", "No"},
    "Gender"        : {"Male", "Female"},
    "MaritalStatus" : {"Single", "Married", "Divorced"},
    "BusinessTravel": {"Travel_Rarely", "Travel_Frequently", "Non-Travel"},
}
for col, esperado in CATEGORIAS_ESPERADAS.items():
    if col in df_raw.columns:
        inesperado = set(df_raw[col].dropna().unique()) - esperado
        if inesperado:
            avisos.append(f"{col}: categorias inesperadas → {inesperado}")

print("RELATÓRIO DE VALIDAÇÃO")
if erros:
    for e in erros:
        print(f"   ERRO  : {e}")
    sys.exit("Pipeline interrompido — corrija os erros acima.")
else:
    print("   Nenhum erro crítico encontrado.")

if avisos:
    for a in avisos:
        print(f"    AVISO : {a}")
else:
    print("   Nenhum aviso.")


RELATÓRIO DE VALIDAÇÃO
   Nenhum erro crítico encontrado.
   Nenhum aviso.



## 3. Transformação

Trabalha sobre uma cópia do DataFrame bruto. As etapas são:

1. Remover colunas constantes / sem valor analítico  
2. Renomear para português (snake_case)  
3. Converter tipos  
4. Decodificar escalas ordinais  
5. Criar flags binárias (0/1)  
6. Padronizar textos

In [24]:
df = df_raw.copy()

# 3.1  Remover colunas constantes / sem utilidade analítica
COLUNAS_DESCARTAR = ["EmployeeCount", "Over18", "StandardHours"]
df.drop(columns=[c for c in COLUNAS_DESCARTAR if c in df.columns], inplace=True)
print(f"[3.1] Colunas removidas: {COLUNAS_DESCARTAR}")


[3.1] Colunas removidas: ['EmployeeCount', 'Over18', 'StandardHours']


In [25]:
# 3.2  Renomear colunas (inglês → português, snake_case)
MAPA_COLUNAS = {
    "Age"                     : "idade",
    "Attrition"               : "attrition",
    "BusinessTravel"          : "frequencia_viagem",
    "DailyRate"               : "taxa_diaria",
    "Department"              : "departamento",
    "DistanceFromHome"        : "distancia_casa_km",
    "Education"               : "nivel_educacao_cod",
    "EducationField"          : "area_formacao",
    "EmployeeNumber"          : "id_funcionario",
    "EnvironmentSatisfaction" : "satisfacao_ambiente_cod",
    "Gender"                  : "genero",
    "HourlyRate"              : "taxa_horaria",
    "JobInvolvement"          : "envolvimento_trabalho_cod",
    "JobLevel"                : "nivel_cargo_cod",
    "JobRole"                 : "cargo",
    "JobSatisfaction"         : "satisfacao_trabalho_cod",
    "MaritalStatus"           : "estado_civil",
    "MonthlyIncome"           : "renda_mensal",
    "MonthlyRate"             : "taxa_mensal",
    "NumCompaniesWorked"      : "qtd_empresas_anteriores",
    "OverTime"                : "horas_extras",
    "PercentSalaryHike"       : "perc_aumento_salarial",
    "PerformanceRating"       : "avaliacao_desempenho_cod",
    "RelationshipSatisfaction": "satisfacao_relacionamento_cod",
    "StockOptionLevel"        : "nivel_stock_option",
    "TotalWorkingYears"       : "anos_experiencia_total",
    "TrainingTimesLastYear"   : "treinamentos_ultimo_ano",
    "WorkLifeBalance"         : "equilibrio_vida_trabalho_cod",
    "YearsAtCompany"          : "anos_empresa",
    "YearsInCurrentRole"      : "anos_cargo_atual",
    "YearsSinceLastPromotion" : "anos_desde_ultima_promocao",
    "YearsWithCurrManager"    : "anos_mesmo_gestor",
}
df.rename(columns=MAPA_COLUNAS, inplace=True)
print(f"[3.2] Colunas renomeadas: {len(MAPA_COLUNAS)}")
df.columns.tolist()


[3.2] Colunas renomeadas: 32


['idade',
 'attrition',
 'frequencia_viagem',
 'taxa_diaria',
 'departamento',
 'distancia_casa_km',
 'nivel_educacao_cod',
 'area_formacao',
 'id_funcionario',
 'satisfacao_ambiente_cod',
 'genero',
 'taxa_horaria',
 'envolvimento_trabalho_cod',
 'nivel_cargo_cod',
 'cargo',
 'satisfacao_trabalho_cod',
 'estado_civil',
 'renda_mensal',
 'taxa_mensal',
 'qtd_empresas_anteriores',
 'horas_extras',
 'perc_aumento_salarial',
 'avaliacao_desempenho_cod',
 'satisfacao_relacionamento_cod',
 'nivel_stock_option',
 'anos_experiencia_total',
 'treinamentos_ultimo_ano',
 'equilibrio_vida_trabalho_cod',
 'anos_empresa',
 'anos_cargo_atual',
 'anos_desde_ultima_promocao',
 'anos_mesmo_gestor']

In [26]:
# 3.3  Converter tipos

# Categóricas nominais → pd.Categorical 
CAT_NOMINAIS = [
    "frequencia_viagem", "departamento", "area_formacao",
    "genero", "cargo", "estado_civil",
]
for col in CAT_NOMINAIS:
    df[col] = pd.Categorical(df[col])

# Flags Yes/No → bool
FLAGS_BINARIAS = ["attrition", "horas_extras"]
for col in FLAGS_BINARIAS:
    df[col] = df[col].map({"Yes": True, "No": False}).astype("boolean")

print("[3.3] Tipos convertidos.")
df.dtypes


[3.3] Tipos convertidos.


idade                               int64
attrition                         boolean
frequencia_viagem                category
taxa_diaria                         int64
departamento                     category
distancia_casa_km                   int64
nivel_educacao_cod                  int64
area_formacao                    category
id_funcionario                      int64
satisfacao_ambiente_cod             int64
genero                           category
taxa_horaria                        int64
envolvimento_trabalho_cod           int64
nivel_cargo_cod                     int64
cargo                            category
satisfacao_trabalho_cod             int64
estado_civil                     category
renda_mensal                        int64
taxa_mensal                         int64
qtd_empresas_anteriores             int64
horas_extras                      boolean
perc_aumento_salarial               int64
avaliacao_desempenho_cod            int64
satisfacao_relacionamento_cod     

In [27]:
# 3.4  Decodificar escalas ordinais → rótulos legíveis

ESCALA_SATISFACAO = {1: "Baixo", 2: "Médio", 3: "Alto", 4: "Muito Alto"}
ESCALA_LIKERT_4   = {1: "Ruim",  2: "Regular", 3: "Bom", 4: "Excelente"}

MAPA_ORDINAIS = {
    "nivel_educacao_cod": {
        1: "Ensino Médio", 2: "Técnico", 3: "Graduação",
        4: "Mestrado",     5: "Doutorado",
    },
    "satisfacao_ambiente_cod"      : ESCALA_SATISFACAO,
    "envolvimento_trabalho_cod"    : ESCALA_SATISFACAO,
    "satisfacao_trabalho_cod"      : ESCALA_SATISFACAO,
    "satisfacao_relacionamento_cod": ESCALA_SATISFACAO,
    "equilibrio_vida_trabalho_cod" : {
        1: "Ruim", 2: "Regular", 3: "Bom", 4: "Ótimo"
    },
    "avaliacao_desempenho_cod"     : {3: "Excelente", 4: "Excepcional"},
    "nivel_cargo_cod"              : {
        1: "Júnior", 2: "Pleno", 3: "Sênior", 4: "Especialista", 5: "Diretor"
    },
}

ORDENS = {
    "nivel_educacao_cod"      : ["Ensino Médio","Técnico","Graduação","Mestrado","Doutorado"],
    "satisfacao_ambiente_cod" : ["Baixo","Médio","Alto","Muito Alto"],
    "envolvimento_trabalho_cod": ["Baixo","Médio","Alto","Muito Alto"],
    "satisfacao_trabalho_cod" : ["Baixo","Médio","Alto","Muito Alto"],
    "satisfacao_relacionamento_cod": ["Baixo","Médio","Alto","Muito Alto"],
    "equilibrio_vida_trabalho_cod" : ["Ruim","Regular","Bom","Ótimo"],
    "avaliacao_desempenho_cod"     : ["Excelente","Excepcional"],
    "nivel_cargo_cod"              : ["Júnior","Pleno","Sênior","Especialista","Diretor"],
}

for cod_col, mapa in MAPA_ORDINAIS.items():
    novo_col = cod_col.replace("_cod", "")
    ordem    = ORDENS.get(cod_col)
    df[novo_col] = pd.Categorical(
        df[cod_col].map(mapa),
        categories=ordem,
        ordered=True,
    )

print("[3.4] Ordinais decodificadas.")
# Verificação rápida
colunas_novas = [c.replace("_cod","") for c in MAPA_ORDINAIS]
df[colunas_novas].head(3)


[3.4] Ordinais decodificadas.


,nivel_educacao,satisfacao_ambiente,envolvimento_trabalho,satisfacao_trabalho,satisfacao_relacionamento,equilibrio_vida_trabalho,avaliacao_desempenho,nivel_cargo
0,Técnico,Médio,Alto,Muito Alto,Baixo,Ruim,Excelente,Pleno
1,Ensino Médio,Alto,Médio,Médio,Muito Alto,Bom,Excepcional,Pleno
2,Técnico,Muito Alto,Médio,Alto,Médio,Bom,Excelente,Júnior


In [28]:
# 3.5  Padronizar textos categóricos

# Frequência de viagem → rótulos em português
df["frequencia_viagem"] = df["frequencia_viagem"].map({
    "Non-Travel"        : "Sem Viagem",
    "Travel_Rarely"     : "Viaja Raramente",
    "Travel_Frequently" : "Viaja Frequentemente",
})

# Gênero
df["genero"] = df["genero"].map({"Male": "Masculino", "Female": "Feminino"})

# Estado civil
df["estado_civil"] = df["estado_civil"].map({
    "Single"  : "Solteiro(a)",
    "Married" : "Casado(a)",
    "Divorced": "Divorciado(a)",
})

print("[3.5] Textos padronizados.")


[3.5] Textos padronizados.


## 4. Enriquecimento

Criação de métricas derivadas e features prontas para uso nos visuais do Power BI.

In [29]:
# 4.1  Flags de risco (baseadas nos padrões encontrados na EDA)

# Risco de attrition — combinação de fatores observados
df["flag_horas_extras"]          = df["horas_extras"].astype(bool)
df["flag_sem_promocao_3a"]       = df["anos_desde_ultima_promocao"] >= 3
df["flag_distancia_alta"]        = df["distancia_casa_km"] > 15
df["flag_satisfacao_baixa"]      = df["satisfacao_trabalho_cod"] <= 2
df["flag_renda_abaixo_mediana"]  = df["renda_mensal"] < df["renda_mensal"].median()

# Score de risco composto (0–5): soma das flags
FATORES_RISCO = [
    "flag_horas_extras",
    "flag_sem_promocao_3a",
    "flag_distancia_alta",
    "flag_satisfacao_baixa",
    "flag_renda_abaixo_mediana",
]
df["score_risco"] = df[FATORES_RISCO].sum(axis=1).astype(int)

df["nivel_risco"] = pd.cut(
    df["score_risco"],
    bins=[-1, 1, 3, 5],
    labels=["Baixo", "Médio", "Alto"],
).astype("category")

print("[4.1] Flags e score de risco criados.")
df[["id_funcionario","score_risco","nivel_risco"] + FATORES_RISCO].head(5)


[4.1] Flags e score de risco criados.


,id_funcionario,score_risco,nivel_risco,flag_horas_extras,flag_sem_promocao_3a,flag_distancia_alta,flag_satisfacao_baixa,flag_renda_abaixo_mediana
0,1,1,Baixo,True,False,False,False,False
1,2,1,Baixo,False,False,False,True,False
2,4,2,Médio,True,False,False,False,True
3,5,3,Médio,True,True,False,False,True
4,7,2,Médio,False,False,False,True,True


In [30]:
# 4.2  Faixas etárias e de renda

df["faixa_etaria"] = pd.cut(
    df["idade"],
    bins=[17, 25, 35, 45, 55, 99],
    labels=["18–25", "26–35", "36–45", "46–55", "55+"],
    ordered=True,
)

QUARTIS_RENDA = df["renda_mensal"].quantile([0.25, 0.50, 0.75]).values
df["faixa_renda"] = pd.cut(
    df["renda_mensal"],
    bins=[0, QUARTIS_RENDA[0], QUARTIS_RENDA[1], QUARTIS_RENDA[2], 99999],
    labels=["Q1 — Baixa", "Q2 — Média-Baixa", "Q3 — Média-Alta", "Q4 — Alta"],
    ordered=True,
)

print("[4.2] Faixas etária e de renda criadas.")
print(f"  Quartis de renda: {QUARTIS_RENDA.astype(int).tolist()}")


[4.2] Faixas etária e de renda criadas.
  Quartis de renda: [2911, 4919, 8379]


In [31]:
# 4.3  Métricas de tenure e estabilidade

# Proporção da carreira na empresa atual
df["perc_carreira_empresa"] = (
    df["anos_empresa"] / df["anos_experiencia_total"].replace(0, np.nan)
).clip(0, 1).round(4)

# Tempo médio por empresa anterior (excluindo empresa atual)
anos_fora = df["anos_experiencia_total"] - df["anos_empresa"]
df["media_anos_por_empresa_anterior"] = (
    (anos_fora / df["qtd_empresas_anteriores"].replace(0, np.nan))
    .clip(0)
    .round(2)
)

# Estagnação no cargo atual: anos no cargo / anos na empresa
df["indice_estagnacao_cargo"] = (
    df["anos_cargo_atual"] / df["anos_empresa"].replace(0, np.nan)
).clip(0, 1).round(4)

print("[4.3] Métricas de tenure criadas.")
df[["id_funcionario","perc_carreira_empresa",
    "media_anos_por_empresa_anterior","indice_estagnacao_cargo"]].head(4)


[4.3] Métricas de tenure criadas.


,id_funcionario,perc_carreira_empresa,media_anos_por_empresa_anterior,indice_estagnacao_cargo
0,1,0.75,0.25,0.67
1,2,1.00,0.00,0.70
2,4,0.00,1.17,NaN
3,5,1.00,0.00,0.88


In [32]:
# 4.4  Custo estimado de reposição 

FATOR_REPOSICAO = {
    "Júnior"      : 0.50,
    "Pleno"       : 0.75,
    "Sênior"      : 1.00,
    "Especialista": 1.50,
    "Diretor"     : 2.00,
}

df["custo_reposicao_est"] = (
    df["nivel_cargo"].astype(str)
    .map(FATOR_REPOSICAO)
    .fillna(0.75)
    * df["renda_mensal"] * 12
).round(0).astype(int)

print("[4.4] Custo de reposição estimado.")
df.groupby("nivel_cargo")["custo_reposicao_est"].mean().map("${:,.0f}".format)


[4.4] Custo de reposição estimado.


nivel_cargo
Júnior           $16,721
Pleno            $49,520
Sênior          $117,807
Especialista    $279,068
Diretor         $460,604
Name: custo_reposicao_est, dtype: str

---
## 5. Verificação pós-transformação

In [33]:
print(f"Shape final: {df.shape[0]:,} linhas  ×  {df.shape[1]} colunas")
print()
print("Tipos por grupo:")
print(df.dtypes.value_counts().to_string())
print()
print("Nulos por coluna (somente com nulos > 0):")
nulos_final = df.isnull().sum()
nulos_final = nulos_final[nulos_final > 0]
if nulos_final.empty:
    print("  Nenhum nulo encontrado ")
else:
    print(nulos_final.to_string())
print()
print("Distribuição attrition:")
print(df["attrition"].value_counts().to_string())
print()
print("Distribuição nível de risco:")
print(df["nivel_risco"].value_counts().sort_index().to_string())


Shape final: 1,470 linhas  ×  53 colunas

Tipos por grupo:
int64       26
bool         5
category     4
float64      3
boolean      2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1

Nulos por coluna (somente com nulos > 0):
perc_carreira_empresa               11
media_anos_por_empresa_anterior    197
indice_estagnacao_cargo             44

Distribuição attrition:
attrition
False    1233
True      237

Distribuição nível de risco:
nivel_risco
Baixo    672
Médio    757
Alto      41


In [34]:
# Estatísticas descritivas das métricas numéricas principais
df[[
    "idade", "renda_mensal", "anos_empresa",
    "anos_experiencia_total", "score_risco", "custo_reposicao_est",
]].describe().T.round(2)


,count,mean,std,min,25%,50%,75%,max
idade,"1,470.00",36.92,9.14,18.00,30.00,36.00,43.00,60.00
renda_mensal,"1,470.00","6,502.93","4,707.96","1,009.00","2,911.00","4,919.00","8,379.00","19,999.00"
anos_empresa,"1,470.00",7.01,6.13,0.00,3.00,5.00,9.00,40.00
anos_experiencia_total,"1,470.00",11.28,7.78,0.00,6.00,10.00,15.00,40.00
score_risco,"1,470.00",1.65,0.97,0.00,1.00,2.00,2.00,5.00
custo_reposicao_est,"1,470.00","83,379.95","109,090.63","6,054.00","17,742.00","44,091.00","90,552.00","479,976.00"


---
## 6. Exportação para Power BI

São geradas **três tabelas** seguindo um modelo estrela simplificado:

| Tabela | Tipo | Descrição |
|--------|------|-----------|
| `fato_funcionarios.csv` | Fato | Uma linha por funcionário — métricas e chaves |
| `dim_cargos.csv` | Dimensão | Atributos de cargo e departamento |
| `dim_satisfacao.csv` | Dimensão | Scores de satisfação e bem-estar por funcionário |


In [35]:
# 6.1  Tabela fato — métricas principais por funcionário

COLUNAS_FATO = [
    "id_funcionario",
    "idade", "faixa_etaria", "genero", "estado_civil",
    "departamento", "cargo", "nivel_cargo", "nivel_cargo_cod",
    "area_formacao", "nivel_educacao",
    "renda_mensal", "faixa_renda", "taxa_diaria", "taxa_horaria",
    "perc_aumento_salarial", "nivel_stock_option",
    "anos_empresa", "anos_cargo_atual", "anos_desde_ultima_promocao",
    "anos_mesmo_gestor", "anos_experiencia_total", "qtd_empresas_anteriores",
    "horas_extras", "frequencia_viagem", "distancia_casa_km",
    "treinamentos_ultimo_ano",
    "perc_carreira_empresa", "media_anos_por_empresa_anterior",
    "indice_estagnacao_cargo", "custo_reposicao_est",
    "score_risco", "nivel_risco",
    *FATORES_RISCO,
    "attrition",
]

fato = df[[c for c in COLUNAS_FATO if c in df.columns]].copy()

# Converte booleans para melhor compatibilidade com Power BI
for col in fato.select_dtypes(include="boolean").columns:
    fato[col] = fato[col].astype("Int64")

fato.to_csv(f"{OUTPUT_DIR}/fato_funcionarios.csv", index=False, encoding="utf-8-sig")
print(f"[6.1] fato_funcionarios.csv  → {fato.shape[0]:,} linhas × {fato.shape[1]} colunas")


[6.1] fato_funcionarios.csv  → 1,470 linhas × 39 colunas


In [36]:
# 6.2  Dimensão cargo — tabela de referência para slicers

dim_cargos = (
    df[["cargo", "departamento", "nivel_cargo", "nivel_cargo_cod"]]
    .drop_duplicates()
    .sort_values(["departamento", "nivel_cargo_cod"])
    .reset_index(drop=True)
)
dim_cargos.insert(0, "id_cargo", dim_cargos.index + 1)

dim_cargos.to_csv(f"{OUTPUT_DIR}/dim_cargos.csv", index=False, encoding="utf-8-sig")
print(f"[6.2] dim_cargos.csv         → {dim_cargos.shape[0]:,} linhas × {dim_cargos.shape[1]} colunas")
dim_cargos


[6.2] dim_cargos.csv         → 31 linhas × 5 colunas


,id_cargo,cargo,departamento,nivel_cargo,nivel_cargo_cod
0,1,Human Resources,Human Resources,Júnior,1
1,2,Human Resources,Human Resources,Pleno,2
2,3,Human Resources,Human Resources,Sênior,3
3,4,Manager,Human Resources,Especialista,4
4,5,Manager,Human Resources,Diretor,5
5,6,Laboratory Technician,Research & Development,Júnior,1
6,7,Research Scientist,Research & Development,Júnior,1
7,8,Research Scientist,Research & Development,Pleno,2
8,9,Healthcare Representative,Research & Development,Pleno,2
9,10,Laboratory Technician,Research & Development,Pleno,2


In [37]:
# 6.3  Dimensão satisfação — scores de bem-estar por funcionário

COLUNAS_SAT = [
    "id_funcionario",
    "satisfacao_trabalho", "satisfacao_trabalho_cod",
    "satisfacao_ambiente", "satisfacao_ambiente_cod",
    "satisfacao_relacionamento", "satisfacao_relacionamento_cod",
    "envolvimento_trabalho", "envolvimento_trabalho_cod",
    "equilibrio_vida_trabalho", "equilibrio_vida_trabalho_cod",
    "avaliacao_desempenho", "avaliacao_desempenho_cod",
]

dim_satisfacao = df[[c for c in COLUNAS_SAT if c in df.columns]].copy()

# Score composto de bem-estar (média dos 5 índices de satisfação, escala 1–4)
indices_numericos = [
    "satisfacao_trabalho_cod", "satisfacao_ambiente_cod",
    "satisfacao_relacionamento_cod", "envolvimento_trabalho_cod",
    "equilibrio_vida_trabalho_cod",
]
dim_satisfacao["score_bemestar"] = (
    df[indices_numericos].mean(axis=1).round(2)
)

dim_satisfacao.to_csv(f"{OUTPUT_DIR}/dim_satisfacao.csv", index=False, encoding="utf-8-sig")
print(f"[6.3] dim_satisfacao.csv     → {dim_satisfacao.shape[0]:,} linhas × {dim_satisfacao.shape[1]} colunas")
dim_satisfacao.head(4)


[6.3] dim_satisfacao.csv     → 1,470 linhas × 14 colunas


,id_funcionario,satisfacao_trabalho,satisfacao_trabalho_cod,satisfacao_ambiente,satisfacao_ambiente_cod,satisfacao_relacionamento,satisfacao_relacionamento_cod,envolvimento_trabalho,envolvimento_trabalho_cod,equilibrio_vida_trabalho,equilibrio_vida_trabalho_cod,avaliacao_desempenho,avaliacao_desempenho_cod,score_bemestar
0,1,Muito Alto,4,Médio,2,Baixo,1,Alto,3,Ruim,1,Excelente,3,2.20
1,2,Médio,2,Alto,3,Muito Alto,4,Médio,2,Bom,3,Excepcional,4,2.80
2,4,Alto,3,Muito Alto,4,Médio,2,Médio,2,Bom,3,Excelente,3,2.80
3,5,Alto,3,Muito Alto,4,Alto,3,Alto,3,Bom,3,Excelente,3,3.20


In [38]:
# 6.4  carga — log da execução

from datetime import datetime

resumo = pd.DataFrame([
    {"tabela": "fato_funcionarios", "linhas": len(fato),
     "colunas": fato.shape[1], "arquivo": "fato_funcionarios.csv"},
    {"tabela": "dim_cargos",        "linhas": len(dim_cargos),
     "colunas": dim_cargos.shape[1], "arquivo": "dim_cargos.csv"},
    {"tabela": "dim_satisfacao",    "linhas": len(dim_satisfacao),
     "colunas": dim_satisfacao.shape[1], "arquivo": "dim_satisfacao.csv"},
])
resumo["gerado_em"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

resumo.to_csv(f"{OUTPUT_DIR}/_resumo_carga.csv", index=False, encoding="utf-8-sig")

print("=" * 60)
print("ETL CONCLUÍDO COM SUCESSO")
print("=" * 60)
print(f"Arquivos gerados em: ./{OUTPUT_DIR}/")
print()
print(resumo[["tabela","linhas","colunas","arquivo"]].to_string(index=False))
print()
print(f"Gerado em: {resumo['gerado_em'].iloc[0]}")

ETL CONCLUÍDO COM SUCESSO
Arquivos gerados em: ./output_pbi/

           tabela  linhas  colunas               arquivo
fato_funcionarios    1470       39 fato_funcionarios.csv
       dim_cargos      31        5        dim_cargos.csv
   dim_satisfacao    1470       14    dim_satisfacao.csv

Gerado em: 2026-06-12 15:33:15
